# Gaussian quadrature

**_A new quadrature module and an update on the function approximation roadmap_**

Jared Callaham • 24 Jul 2026

---

Release v0.5.0 is out today, and with it a new [`quadrature`](#archimedes.quadrature) module including support for Gaussian quadrature implementations that are compatible with Archimedes' symbolic tracing, autodiff, and code generation.

Of course, you could always have just called SciPy yourself to compute the weights and nodes and then done `np.dot(f(x), w)` in an Archimedes-traced function.
The reason there's a quadrature module at all is to begin to introduce some new abstractions that will eventually become the foundation for function approximation functionality loosely inspired by [ApproxFun.jl](https://juliaapproximation.github.io/ApproxFun.jl/stable/) and [FEniCS/Firedrake's UFL](https://docs.fenicsproject.org/ufl/main/manual/introduction.html).
While I think the high-level quadrature functions I'll introduce below should be more or less in their final form, it's possible some of the structure of the quadrature module will change once the function approximation infrastructure starts to fill in.
But as always, updates and breaking changes will be documented and will respect versioning in releases.

With that in mind, I wanted to give a quick tour of the new quadrature capabilities and sketch out where the function approximation work is heading.

## Gaussian quadrature

Gaussian quadrature approximates a weighted integral with a discrete sum over (generally non-uniform) nodes and weights:

$$
\int_a^b f(x) \, w(x) \, dx \approx \sum_{i=1}^n w_i f(x_i),
$$

where $f(x)$ is the function to be integrated, $w(x)$ is a weight function, and $\{x_i\}, \{w_i\}$ are the nodes and weights, which are uniquely determined by the family and order of the quadrature rule.

The most common quadrature family, [Gauss-Legendre quadrature](https://en.wikipedia.org/wiki/Gauss%E2%80%93Legendre_quadrature) uses a domain of $[-1, 1]$, with uniform weight $w(x) = 1$:

$$
\int_{-1}^{1} f(x) \, dx \approx \sum_{i=1}^n w_i f(x_i),
$$

which can be shifted to an arbitrary (finite) domain $[a, b]$ by rescaling the Gauss-Legendre nodes and weights by:

\begin{aligned}
x_i &\leftarrow \frac{b-a}{2} x_i + \frac{a+b}{2} \\
w_i &\leftarrow \frac{b-a}{2} w_i
\end{aligned}

The power of Gaussian quadrature lies in carefully chosen nodes and weights which lead to highly accurate approximations of polynomials (and hence arbitrary smooth functions) with relatively few sample points.
The nodes are the roots of classical orthogonal polynomials associated with the weight function (i.e. Legendre polynomials for $w(x) = 1$ on a finite interval), and the weights are derived from Lagrange interpolation of the nodal data.

Since the nodes and weights on the reference domain can be statically computed, under the hood we use SciPy's `roots_legendre/jacobi/laguerre/hermite` functions to do the actual math.

There are two internal abstractions that keep track of the weight function, reference domain, and reference nodes/weights.
The first is [`Measure`](#archimedes.polynomial.orthogonal.Measure), which combines a weight function with a reference interval to define families of orthogonal polynomials.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import archimedes as arc